<a href="https://colab.research.google.com/github/luizgpereira23-ops/Senai-Projects/blob/main/cadastro_clientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio da Aprendizagem
Desenvolver um programa estruturado que permita:

• Registrar vendas de produtos;

• Armazenar dados em variáveis e estruturas adequadas;

• Calcular totais, médias e identificar melhores resultados;

• Apresentar informações organizadas ao usuário;

• Garantir clareza, correção lógica e boa estrutura do código

## Funcionalidades mínimas:
### 1. Exiba um menu de opções, como:
o Cadastrar venda

o Exibir resumo das vendas

o Mostrar total faturado

o Identificar maior e menor venda

o Encerrar sistema

### 2. Solicite:
o Nome do produto

o Quantidade vendida

o Valor unitário

### 3. Armazene os dados em vetores.
### 4. Realize cálculos automáticos:
o Valor total por venda

o Faturamento total

o Média de vendas

### 5. Utilize estruturas de repetição para permitir múltiplos cadastros.
### 6. Utilize funções para organizar o código (ex.: cadastrarVenda(), calcularTotal(),
mostrarMenu()).


In [58]:
import pandas as pd
import sqlite3 as sql
import matplotlib.pyplot as plt
import seaborn as sns
import os

def cadastrar_venda():
  produto = []
  quantidade = []
  valor_unitario = []

  # Inputs do usuário
  try:
    produto.append(input('Digite o nome do produto: '))
    quantidade.append(int(input('Digite a quantidade vendida: ')))
    valor_unitario.append(float(input('Digite o valor unitário (use . para decimais): ')))
  except Exception as e:
    print(f'Erro: {e}')

  # A criação da DataFrame `new_rows` foi movida para fora do loop
  # para garantir que todas as vendas registradas sejam acumuladas.
  new_rows = pd.DataFrame({'produto':produto,
                          'quantidade':quantidade,
                          'valor_unitario':valor_unitario})

  return new_rows

def criar_relatorio(table_vendas:pd.DataFrame):
  '''
  table_vendas: pd.DataFrame - Tabela de vendas
  - colunas: produto, quantidade, valor_unitario, valor_total

  Calcular:
  - Faturamento total
  - Média de vendas
  - Maior venda
  - Menor venda
  '''
  faturamento_total = table_vendas['valor_total'].sum()
  maior_venda = table_vendas['valor_total'].max()
  menor_venda = table_vendas['valor_total'].min()
  media_vendas = table_vendas['valor_total'].mean()

  # Create a DataFrame with a single row for the report
  relatorio = pd.DataFrame({
      'faturamento_total': [faturamento_total],
      'media_vendas': [media_vendas],
      'maior_venda': [maior_venda],
      'menor_venda': [menor_venda]
  })
  display(relatorio)
  return relatorio

def salvar_relatorio(table_vendas:pd.DataFrame):
  #Salvar a tabela para histórico
  if os.path.isfile('table_vendas.csv'):
    pass
  else:
    table_vendas.to_csv('table_vendas.csv', index=False)

def main():
  if os.path.isfile('table_vendas.csv'):
      table_vendas = pd.read_csv('table_vendas.csv')
  else:
    table_vendas = pd.DataFrame(columns=['produto', 'quantidade', 'valor_unitario'])
  while True:
    input_user = input('''
                          Relatório de vendas\n Digite a opção que deseja visualizar:
                          0 - Sair
                          1 - Cadastrar venda
                          2 - Exibir relatório
                          3 - Dashboard
                          ''')
    if input_user == '0':
      break

    elif input_user == '1':
      nova_linha = cadastrar_venda(table_vendas)
      table_vendas = pd.concat([table_vendas, nova_linha],axis=0, ignore_index=True)
      table_vendas['valor_total'] = table_vendas['quantidade'] * table_vendas['valor_unitario']

      print('\n============Tabela de vendas============')
      display(table_vendas)

      print('\n============Relatório============')
      relatorio = criar_relatorio(table_vendas)
      display(relatorio)
      salvar_relatorio(table_vendas)

    elif input_user == '2':
      print('\n============Relatório de vendas============')
      criar_relatorio(table_vendas)
      salvar_relatorio(table_vendas)

    elif input_user == '3':
      print('\n============Dashboard============')
      # Vendas por produto
      sns.barplot(x='produto', y='quantidade', data=table_vendas)
      plt.show()

  # 'Dashboard'


if __name__ == '__main__':
  main()

Digite o nome do produto: a
Digite a quantidade vendida: 1
Digite o valor unitário (use . para decimais): 1

Cadastrar nova venda? (s/n)n

======Tabela de vendas======


,produto,quantidade,valor_unitario,valor_total
0,a,1,2.0,2.0
1,b,1,1.0,1.0
2,a,1,1.0,1.0
3,b,1,1.0,1.0
4,1,1,1.0,1.0
5,a,1,1.0,1.0
6,a,1,1.0,1.0



======Relatório======


,faturamento_total,media_vendas,maior_venda,menor_venda
0,8.0,1.142857,2.0,1.0


# Kelvin

In [59]:
import sqlite3

conn = sqlite3.connect('estoque.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Produtos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome TEXT NOT NULL,
        quantidade INTEGER NOT NULL CHECK (quantidade >= 0),
        preco REAL NOT NULL CHECK (preco >= 0)
    )
''')

def adicionar_produto():
    nome = input("Nome do produto: ")
    quantidade = int(input("Quantidade: "))
    preco = float(input("Preço: "))
    cursor.execute('INSERT INTO Produtos (nome, quantidade, preco) VALUES (?, ?, ?)', (nome, quantidade, preco))
    conn.commit()
    print("Produto adicionado com sucesso!")

def listar_produtos():
    cursor.execute('SELECT * FROM Produtos')
    produtos = cursor.fetchall()
    print("Produtos em estoque:")
    for produto in produtos:
        print(f"ID: {produto[0]}, Nome: {produto[1]}, Quantidade: {produto[2]}, Preço: {produto[3]:.2f}")

def atualizar_produto():
    id_produto = int(input("ID do produto a ser atualizado: "))
    quantidade = int(input("Nova quantidade (deixe em branco para não alterar): ") or "0")
    preco = float(input("Novo preço (deixe em branco para não alterar): ") or "0")
    if quantidade > 0:
        cursor.execute('UPDATE Produtos SET quantidade = ? WHERE id = ?', (quantidade, id_produto))
    if preco > 0:
        cursor.execute('UPDATE Produtos SET preco = ? WHERE id = ?', (preco, id_produto))
    conn.commit()
    print("Produto atualizado com sucesso!")

def remover_produto():
    id_produto = int(input("ID do produto a ser removido: "))
    cursor.execute('DELETE FROM Produtos WHERE id = ?', (id_produto,))
    conn.commit()
    print("Produto removido com sucesso!")

def buscar_produto():
    busca = input("Buscar produto por ID ou nome: ")
    if busca.isdigit():
        cursor.execute('SELECT * FROM Produtos WHERE id = ?', (int(busca),))
    else:
        cursor.execute('SELECT * FROM Produtos WHERE nome LIKE ?', ('%' + busca + '%',))
    produtos = cursor.fetchall()
    print("Produtos encontrados:")
    for produto in produtos:
        print(f"ID: {produto[0]}, Nome: {produto[1]}, Quantidade: {produto[2]}, Preço: {produto[3]:.2f}")

def calcular_valor_total():
    cursor.execute('SELECT SUM(quantidade * preco) FROM Produtos')
    total = cursor.fetchone()[0]
    print(f"Valor total do estoque: {total:.2f}")

def vender_produto():
    id_produto = int(input("ID do produto a ser vendido: "))
    quantidade_vendida = int(input("Quantidade vendida: "))
    cursor.execute('SELECT quantidade, preco FROM Produtos WHERE id = ?', (id_produto,))
    produto = cursor.fetchone()
    if produto and produto[0] >= quantidade_vendida:
        valor_venda = quantidade_vendida * produto[1]
        cursor.execute('UPDATE Produtos SET quantidade = ? WHERE id = ?', (produto[0] - quantidade_vendida, id_produto))
        conn.commit()
        print(f"Venda realizada com sucesso! Valor da venda: {valor_venda:.2f}")
    else:
        print("Erro: quantidade insuficiente em estoque.")

while True:
    print("\nSistema de Controle de Estoque")
    print("1. Adicionar Produto")
    print("2. Listar Produtos")
    print("3. Atualizar Produto")
    print("4. Remover Produto")
    print("5. Buscar Produto")
    print("6. Calcular Valor Total do Estoque")
    print("7. Vender Produto")
    print("0. Sair")
    opcao = input("Escolha uma opção: ")

    if opcao == '1':
        adicionar_produto()
    elif opcao == '2':
        listar_produtos()
    elif opcao == '3':
        atualizar_produto()
    elif opcao == '4':
        remover_produto()
    elif opcao == '5':
        buscar_produto()
    elif opcao == '6':
        calcular_valor_total()
    elif opcao == '7':
        vender_produto()
    elif opcao == '0':
        break
    else:
        print("Opção inválida. Tente novamente.")


Sistema de Controle de Estoque
1. Adicionar Produto
2. Listar Produtos
3. Atualizar Produto
4. Remover Produto
5. Buscar Produto
6. Calcular Valor Total do Estoque
7. Vender Produto
0. Sair
Escolha uma opção: 2
Produtos em estoque:

Sistema de Controle de Estoque
1. Adicionar Produto
2. Listar Produtos
3. Atualizar Produto
4. Remover Produto
5. Buscar Produto
6. Calcular Valor Total do Estoque
7. Vender Produto
0. Sair
Escolha uma opção: 1
Nome do produto: teste
Quantidade: 1
Preço: 1
Produto adicionado com sucesso!

Sistema de Controle de Estoque
1. Adicionar Produto
2. Listar Produtos
3. Atualizar Produto
4. Remover Produto
5. Buscar Produto
6. Calcular Valor Total do Estoque
7. Vender Produto
0. Sair
Escolha uma opção: 2
Produtos em estoque:
ID: 1, Nome: teste, Quantidade: 1, Preço: 1.00

Sistema de Controle de Estoque
1. Adicionar Produto
2. Listar Produtos
3. Atualizar Produto
4. Remover Produto
5. Buscar Produto
6. Calcular Valor Total do Estoque
7. Vender Produto
0. Sair


KeyboardInterrupt: Interrupted by user